In [1]:
print(123)

123


In [42]:
from dotenv import dotenv_values, load_dotenv
from pathlib import Path
import os

project_root = Path("/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh")
env_path = project_root / ".env"
load_dotenv(env_path, override=True)

api_key = os.environ.get("GROQ_API_KEY") or dotenv_values(env_path).get("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY fehlt. Trage ihn in die .env-Datei ein.")

os.environ["GROQ_API_KEY"] = api_key

In [43]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [44]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [45]:
llm("Hey, what's up?")

"Not much! Just here to help with any questions or topics you'd like to discuss. How's your day going so far?"

In [6]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I'm excited you're interested in the course. However, I need a bit more information to help you. Could you please tell me which course you're referring to and when it started? That way, I can provide you with more accurate guidance on whether you can still join or if there are any prerequisites you need to meet.


In [7]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [8]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [9]:
answer = llm(prompt)
print(answer)

Yes, you can join the course now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [46]:
def rag(question):
    if "search" not in globals() or "build_prompt" not in globals():
        raise RuntimeError("search und build_prompt muessen vor rag definiert werden.")

    search_results = globals()["search"](question)
    user_prompt = globals()["build_prompt"](question, search_results)
    return llm(user_prompt)

In [11]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [12]:
len(docs_url)

44

In [13]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [14]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [15]:
documents[1100]

{'id': 'ed90e0f589',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 5. Deploying Machine Learning Models',
 'question': 'Bind for 0.0.0.0:9696 failed: port is already allocated',
 'answer': 'I was getting the following error when I rebuilt the Docker image, although the port was not allocated, and it was working fine.\n\nError message:\n\n```\nError response from daemon: driver failed programming external connectivity on endpoint beautiful_tharp (875be95c7027cebb853a62fc4463d46e23df99e0175be73641269c3d180f7796): Bind for 0.0.0.0:9696 failed: port is already allocated.\n```\n\n\n\nThe issue can be resolved by running the following command:\n\n```bash\ndocker kill $(docker ps -q)\n```\n\nFor more information, refer to the [GitHub issue on Docker for Windows](https://github.com/docker/for-win/issues/2722).'}

### 1.5 SEARCH

In [16]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [17]:
search_result = index.search(
    question, 
    boost_dict ={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'}, 
    num_results=5)

search_result


[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [18]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [19]:
search_results = search(question)

In [20]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

### 1.6 Building prompt

In [21]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [22]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [23]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [24]:
context = build_context(search_results)

In [25]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [26]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [27]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

### 1.7 LLM

In [28]:
response = openai_client.responses.create(
        model="llama-3.3-70b-versatile",
        input =prompt
        )

In [29]:
response.output_text

'According to the course policies, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start learning and submitting homework without any issues, but keep in mind that you won\'t be able to get a certificate if you\'re not following the course with a "live" cohort. To get started, you can refer to the provided resources such as the LLM Zoomcamp docs, general Zoomcamp logistics docs, and the LLM Zoomcamp GitHub repository.'

In [30]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_01kvwmp7hxfwsa62s1vsj76ya8",
  "created_at": 1782298910.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "llama-3.3-70b-versatile",
  "object": "response",
  "output": [
    {
      "id": "resp_01kvwmp7hxfwsv7anmprpvmyks",
      "summary": [],
      "type": "reasoning",
      "content": null,
      "encrypted_content": null,
      "status": "completed"
    },
    {
      "id": "msg_01kvwmp7hxfwt9syeaf2n74k7j",
      "content": [
        {
          "annotations": [],
          "text": "According to the course policies, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start learning and submitting homework without any issues, but keep in mind that you won't be able to get a certificate if you're not following the course with a \"live\" cohort. To get started, you can refer to the provided resources such as the LLM Zoomc

In [31]:
response.output[0]

ResponseReasoningItem(id='resp_01kvwmp7hxfwsv7anmprpvmyks', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed')

In [32]:
response.output_text

'According to the course policies, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start learning and submitting homework without any issues, but keep in mind that you won\'t be able to get a certificate if you\'re not following the course with a "live" cohort. To get started, you can refer to the provided resources such as the LLM Zoomcamp docs, general Zoomcamp logistics docs, and the LLM Zoomcamp GitHub repository.'

In [33]:
response.usage

ResponseUsage(input_tokens=510, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=109, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=619)

In [34]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0008730000000000001

In [35]:
response = openai_client.responses.create( #responses ist newer Name für Completions
        model="llama-3.3-70b-versatile",
        input =prompt
        )

In [36]:
message_history = [
    {"role": "system", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create( #responses ist newer Name für Completions
        model="llama-3.3-70b-versatile",
        input =message_history
        )

In [37]:
response.output_text

'Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start the course and follow the weekly workflow by referring to the provided documentation and resources.'

### LLM Funktion


In [38]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [39]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [40]:
answer = rag(question, model="llama-3.3-70b-versatile")
print(answer)

You can join the course now, but if you want to receive a certificate, you'll need to submit your project while the course is still accepting submissions. To get started, you can follow the weekly workflow outlined in the course materials, which includes watching lesson videos, working through notebooks/code, and submitting homework through the course platform before the deadlines. Keep in mind that to be eligible for a certificate, you'll need to finish the course with a "live" cohort and participate in peer-reviewing projects.


In [41]:
answer = rag("I just discovered the course. Can I join now?", model="llama-3.3-70b-versatile")
print(answer)

Yes, you can join the course now. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions. You can start with the provided documentation and GitHub repository, and follow the weekly workflow as outlined. Keep in mind that to get a certificate, you need to finish the course with a "live" cohort and participate in peer-reviewing projects.
